# 🌿 Crop Disease Detection — GPU Training Pipeline
### Approved 4-Crop Architecture Remaining Training Jobs:
1. **Grape Unified Model** (G1 Niphad + G2 2024 — 7 Classes, 6,203 images)
2. **Chilli Model** (C1 COLD 2024 — 5 Classes, 1,932 images)
3. **Sugarcane Unified Model** (S1 Maharashtra + S2 Large — 11 Classes, 8,926 images)

*(Note: T1 Tomato baseline is already completed at 90.23% accuracy and remains untouched.)*

---
### Instructions for Execution:
- **On Google Colab:** Click `Runtime` -> `Change runtime type` -> select `T4 GPU` -> `Save`.
- **On Kaggle Notebooks:** Under `Notebook options` (right sidebar), set `Accelerator` to `GPU T4 x2`.
- Total expected runtime on a T4 GPU: **~20 to 25 minutes**.

In [ ]:
# 1. Hardware & Environment Verification
import tensorflow as tf
print(f'TensorFlow Version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs Available: {len(gpus)}')
if gpus:
    for g in gpus:
        print(f'  GPU Device: {g.name}')
else:
    print('⚠️ WARNING: No GPU detected! Please enable GPU runtime for fast training.')

In [ ]:
# 2. Clone Repository or Set Project Path
import os

# If running directly in a cloned repo on Colab/Kaggle:
if not os.path.exists('train_experiment.py'):
    !git clone https://github.com/sohail-148/maharashtra-crop-disease-detection.git project
    %cd project

print('Current Working Directory:', os.getcwd())
!ls -lh

In [ ]:
# 3. Check Dataset Availability
# Specify the folder containing the dataset folders (grape_niphad, grape_2024, chilli_cold, etc.)
# If datasets are located in the project root:
DATASET_ROOT = os.getcwd()

# If datasets are mounted from Google Drive or Kaggle Input, update DATASET_ROOT:
# e.g. DATASET_ROOT = '/kaggle/input/crop-disease-datasets'
# e.g. DATASET_ROOT = '/content/drive/MyDrive/CropDiseaseProject'

print('Using Dataset Root:', DATASET_ROOT)
!python train_experiment.py --help

In [ ]:
# 4. Train Job 1 — Grape Unified Model (7 Classes)
# Combines G1 Niphad (Maharashtra) + G2 2024 into a single robust model
!python train_experiment.py --experiment GRAPE --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 5. Train Job 2 — Chilli Model (5 Classes)
# Trains C1 COLD 2024 dataset
!python train_experiment.py --experiment CHILLI --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 6. Train Job 3 — Sugarcane Unified Model (11 Classes)
# Combines S1 Maharashtra + S2 Large into a single comprehensive model
!python train_experiment.py --experiment SUGARCANE --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 7. Display Metrics & Final Evaluation Summary
import pandas as pd
import glob

results = []
for m in ['grape_unified', 'chilli_cold', 'sugarcane_unified']:
    p = f'results/{m}/test_metrics.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        results.append(df)

if results:
    summary_df = pd.concat(results, ignore_index=True)
    print('=== ALL 3 TRAINED MODELS FINAL TEST RESULTS ===')
    display(summary_df)
else:
    print('Results not generated yet.')

In [ ]:
# 8. Package Trained Models & Results for 1-Click Download
import shutil
!zip -r trained_models_results.zip models/ results/
print('Archive created: trained_models_results.zip')

# If running in Google Colab, trigger direct browser download:
try:
    from google.colab import files
    files.download('trained_models_results.zip')
except Exception:
    print('On Kaggle: download trained_models_results.zip from the right-hand Output panel.')